## IMPORTAMOS LAS LIBRERIAS NECESARIAS

In [ ]:
import os
import io
import re
import time
import random
import logging
import requests
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from io import BytesIO
from datetime import date, datetime, timedelta
from google.cloud import storage

## CODIGO PARA EXTRACCION DE DATA SETS DE LICITACIONES

Previamente debemos tener creado y configurado nuestro bucket en Google Cloud para guardar la data directamente allí. Asimismo, contar con una credencial.

In [ ]:
# ---------------------------------------
# Definimos nuestras variables
# ---------------------------------------

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

if not os.getenv("GOOGLE_APPLICATION_CREDENTIALS"):
    raise EnvironmentError(
        "Debes configurar la variable de entorno GOOGLE_APPLICATION_CREDENTIALS con la ruta a tu JSON de GCP"
    )

API_KEY = os.getenv("API_KEY")

if not API_KEY:
    raise EnvironmentError(
        "Debes configurar la variable de entorno API_KEY con tu API de Mercado Público"
    )

BUCKET_NAME = "mp_datos_proyecto"

FULL_PATH      = "data/licitaciones/full"
PROCESSED_PATH = "data/licitaciones/processed/mensual"

client = storage.Client()
bucket = client.bucket(BUCKET_NAME)

In [ ]:
# ---------------------------------------
# Función consulta API con reintentos
# ---------------------------------------

def consultar_api(url, max_retries=5):
    for intento in range(max_retries):
        try:
            response = requests.get(url)

            if response.status_code == 200:
                return response.json()
            else:
                espera = (intento + 1) * 5
                log.warning(f"Error {response.status_code}. Esperando {espera}s...")
                time.sleep(espera)

        except Exception as e:
            log.error(f"Error conexión: {e}")
            time.sleep(5)

    return None

In [ ]:
# ---------------------------------------
# Función generar fechas por mes
# ---------------------------------------

def generar_fechas_mes(year, month):
    inicio = datetime(year, month, 1)

    if month == 12:
        fin = datetime(year + 1, 1, 1) - timedelta(days=1)
    else:
        fin = datetime(year, month + 1, 1) - timedelta(days=1)

    fechas = []
    actual = inicio

    while actual <= fin:
        fechas.append(actual.strftime("%d%m%Y"))
        actual += timedelta(days=1)

    return fechas

In [ ]:
# ---------------------------------------
# Funcion subir a GCS
# ---------------------------------------

def subir_a_gcs(df, ruta_blob):
    buffer = BytesIO()
    df.to_parquet(buffer, index=False)
    buffer.seek(0)

    blob = bucket.blob(ruta_blob)
    blob.upload_from_file(buffer, content_type="application/octet-stream")

In [ ]:
# ---------------------------------------
# Descarga LICITACIONES - año 2025
# ---------------------------------------

# Para evitar saturaciones de la API, manualmente se hará el cambio de año y así obtener data desde 2020 al 2025.

df_licitaciones_total = []

for mes in range(1, 13):

    log.info(f"Procesando MES {mes:02d} - 2025")

    fechas = generar_fechas_mes(2025, mes)

    for i, fecha in enumerate(fechas):

        log.info(f"Día {i+1}/{len(fechas)} → {fecha}")

        url = f"https://api.mercadopublico.cl/servicios/v1/publico/licitaciones.json?fecha={fecha}&ticket={API_KEY}"

        data = consultar_api(url)

        if data and "Listado" in data:

            df_licitaciones = pd.json_normalize(data["Listado"])

            if not df_licitaciones.empty:

                df_licitaciones["anio"] = 2025
                df_licitaciones["mes"]  = mes

                df_licitaciones_total.append(df_licitaciones)

            else:
                log.warning("DataFrame vacío")

        else:
            log.warning("Sin datos")

        time.sleep(random.uniform(1.5, 4.0))

In [ ]:
# ----------------------------------------------------
# Generar data full (licitaciones totales) y subir GCS
# ----------------------------------------------------

log.info("Generando dataset FULL...")

df_licitaciones = pd.concat(df_licitaciones_total, ignore_index=True)

ruta_full = f"{FULL_PATH}/licitaciones_dataset_completo.parquet"
subir_a_gcs(df_licitaciones, ruta_full)

log.info("✔ FULL generado en GCS")

In [ ]:
# -------------------------------------------------------------
# Generar data procesada (licitaciones mensuales) y subir a GCS
# -------------------------------------------------------------

log.info("Generando datasets mensuales...")

for (anio, mes), df_mes in df_licitaciones.groupby(["anio", "mes"]):

    ruta_mes = f"{PROCESSED_PATH}/anio={anio}/licitaciones_{anio}_{mes:02d}.parquet"

    subir_a_gcs(df_mes, ruta_mes)

    log.info(f"✔ PROCESSED: {anio}-{mes:02d}")

## ALGUNAS CONSULTAS

In [ ]:
df_licitaciones.head()

,CodigoExterno,Nombre,CodigoEstado,FechaCierre
0,564162-187-L119,(id.48430) Particulas para Depto de Física,7,2020-01-01T17:04:01.527
1,1058134-381-L119,Medicamentos Enero 2020,7,2020-01-01T15:04:01.643
2,1663-111-L119,SERVICIO DE LABORATORIO DENTAL PARA PRÓTESIS M...,7,2020-01-01T21:44:01.157
3,2713-290-L119,CLASES DIRIGIDAS,7,2020-01-01T19:29:01.22
4,1058133-15-LR19,Convenio de Suministro Farmacos Inyectables Red,8,2019-11-04T15:01:00


In [ ]:
df_licitaciones.columns

Index(['CodigoExterno', 'Nombre', 'CodigoEstado', 'FechaCierre'], dtype='object')

In [ ]:
df_licitaciones.shape

(826238, 4)

In [ ]:
df_licitaciones.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 826238 entries, 0 to 826237
Data columns (total 4 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   CodigoExterno  826238 non-null  object
 1   Nombre         826238 non-null  object
 2   CodigoEstado   826238 non-null  int64 
 3   FechaCierre    826237 non-null  object
dtypes: int64(1), object(3)
memory usage: 25.2+ MB


In [ ]:
df_licitaciones['tipo_licitacion'] = df_licitaciones['CodigoExterno'].str.extract(r'-([A-Z0-9]+)\d{2}$')

# Contamos cuántas veces aparece cada tipo y sacamos el Top 3
top_3_tipos_lic = df_licitaciones['tipo_licitacion'].value_counts().head(3)

print("Los 3 tipos de licitaciones más comunes son:")
print(top_3_tipos_lic)

Los 3 tipos de licitaciones más comunes son:
tipo_licitacion
LE    361115
L1    269241
LP     63356
Name: count, dtype: int64


In [ ]:
df_licitaciones['codigo_organismo'] = df_licitaciones['CodigoExterno'].str.extract(r'^(\d+)-')

# Contamos cuántas veces aparece cada organismo y sacamos el Top 3
top_3_organismos = df_licitaciones['codigo_organismo'].value_counts().head(3)

print("Los 3 organismos más comunes son:")
print(top_3_organismos)

Los 3 organismos más comunes son:
codigo_organismo
621     7483
1658    4888
2409    4552
Name: count, dtype: int64


In [ ]:
df_licitaciones.isnull().sum().sort_values(ascending=False)

tipo_licitacion    12
FechaCierre         1
CodigoExterno       0
CodigoEstado        0
Nombre              0
dtype: int64

## EXTRACCION DE NOMBRES Y REGIONES DE 19 ORGANISMOS ESTRATÉGICOS

In [ ]:
!pip install tenacity

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\PREDATOR\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception_type,
    before_sleep_log,
)

In [ ]:
# ─────────────────────────────────────────────
# CONFIGURACIÓN
# ─────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

if not os.getenv("GOOGLE_APPLICATION_CREDENTIALS"):
    raise EnvironmentError(
        "Debes configurar la variable de entorno GOOGLE_APPLICATION_CREDENTIALS con la ruta a tu JSON de GCP"
    )

API_KEY = os.getenv("API_KEY")

if not API_KEY:
    raise EnvironmentError(
        "Debes configurar la variable de entorno API_KEY con tu API de Mercado Público"
    )

BUCKET_NAME = "mp_datos_proyecto"

# Paths GCS
LICITACIONES_ANUAL_PATH = "data/licitaciones/processed/anual"
CURATED_PATH            = "data/licitaciones/curated/organismos_le"
AUDIT_PATH              = "data/licitaciones/audit/organismos_le"

# Parámetros
YEARS       = list(range(2020, 2026))
TIMEOUT_SEC = 20
SLEEP_SEC   = 1.5   # pausa entre requests (19 en total, sin prisa)

# Prefijos de interés: resultado de aplicar filtros
# (adjudicadas + tipo LE + >= 150 licitaciones)
# Valor del dict = ranking por volumen (referencia, no se usa en el código)
PREFIJOS_INTERES = {
    '3510', '3863', '4857', '3656', '1660', '3960', '3794',
    '2196', '979',  '3589', '2564', '2342', '4809', '1247197',
    '1057503', '3928', '3709', '1509', '3508'
}

In [ ]:
# ─────────────────────────────────────────────
# GCS CLIENT
# ─────────────────────────────────────────────

gcs_client = storage.Client()
bucket     = gcs_client.bucket(BUCKET_NAME)

In [ ]:
# ─────────────────────────────────────────────
# UTILIDADES GCS
# ─────────────────────────────────────────────

def leer_parquet_gcs(gcs_path: str) -> pd.DataFrame:
    blob = bucket.blob(gcs_path)
    if not blob.exists():
        log.warning(f"No existe en GCS: {gcs_path}")
        return pd.DataFrame()
    data = blob.download_as_bytes()
    return pd.read_parquet(io.BytesIO(data))


def upload_parquet_gcs(df: pd.DataFrame, gcs_path: str) -> None:
    """Sube DataFrame a GCS como Parquet Snappy sin escribir en disco."""
    table  = pa.Table.from_pandas(df, preserve_index=False)
    buffer = io.BytesIO()
    pq.write_table(table, buffer, compression="snappy")
    buffer.seek(0)
    blob = bucket.blob(gcs_path)
    blob.upload_from_file(buffer, content_type="application/octet-stream")
    log.info(f"Subido: gs://{BUCKET_NAME}/{gcs_path} ({len(df)} filas)")

In [ ]:
# ─────────────────────────────────────────────
# PASO 1: PREPARAR DATASET DESDE GCS
# ─────────────────────────────────────────────

def extraer_tipo(codigo: str):
    """
    Extrae el tipo de licitación desde CodigoExterno.
    '1658-257-LE22' → 'LE'
    '2704-190-L121' → 'L1'
    """
    match = re.search(r"-([A-Z][A-Z0-9]{0,2})\d{2}$", str(codigo))
    return match.group(1) if match else None


def extraer_prefijo(codigo: str):
    """
    Extrae los números antes del primer guión.
    '1509-5-L114' → '1509'
    '1247197-32-LE22' → '1247197'
    """
    match = re.match(r"^(\d+)-", str(codigo))
    return match.group(1) if match else None


def construir_dataset_filtrado() -> pd.DataFrame:
    """
    Lee parquets anuales de GCS y retorna solo las licitaciones
    LE adjudicadas correspondientes a los 19 prefijos de interés.

    Returns:
        DataFrame con columnas: CodigoExterno, prefijo, tipo, total_le_adjudicadas
    """
    log.info("── PASO 1: Cargando y filtrando dataset desde GCS ──")

    frames = []
    for year in YEARS:
        path = f"{LICITACIONES_ANUAL_PATH}/licitaciones_{year}.parquet"
        df   = leer_parquet_gcs(path)
        if not df.empty:
            log.info(f"  {year}: {len(df):,} filas")
            frames.append(df)

    if not frames:
        raise RuntimeError("Sin datasets anuales en GCS.")

    df_total = pd.concat(frames, ignore_index=True)
    log.info(f"Total combinado: {len(df_total):,} licitaciones")

    # Extraer prefijo y tipo desde CodigoExterno
    df_total["prefijo"] = df_total["CodigoExterno"].apply(extraer_prefijo)
    df_total["tipo"]    = df_total["CodigoExterno"].apply(extraer_tipo)

    # Filtros: adjudicadas + tipo LE + prefijos de interés
    df_filtrado = df_total[
        (df_total["CodigoEstado"] == 8) &
        (df_total["tipo"] == "LE") &
        (df_total["prefijo"].isin(PREFIJOS_INTERES))
    ].copy()

    log.info(f"Licitaciones LE adjudicadas en prefijos de interés: {len(df_filtrado):,}")

    # Conteo por prefijo
    conteo = (
        df_filtrado.groupby("prefijo")
        .size()
        .reset_index(name="total_le_adjudicadas")
        .sort_values("total_le_adjudicadas", ascending=False)
        .reset_index(drop=True)
    )

    log.info(f"\nConteo por prefijo:\n{conteo.to_string(index=False)}\n")

    # Un código representativo por prefijo para consultar la API
    # Tomamos el más reciente (último año disponible) para maximizar
    # probabilidad de respuesta exitosa en el endpoint de detalle
    representativos = (
        df_filtrado.sort_values("FechaCierre", ascending=False)
        .groupby("prefijo")["CodigoExterno"]
        .first()
        .reset_index()
        .rename(columns={"CodigoExterno": "codigo_representativo"})
    )

    log.info(f"Códigos representativos seleccionados:\n{representativos.to_string(index=False)}\n")

    # Unir conteo con representativo
    resultado = conteo.merge(representativos, on="prefijo", how="left")

    return resultado

In [ ]:
# ─────────────────────────────────────────────
# PASO 2: CONSULTAR API POR CÓDIGO INDIVIDUAL
#
# ?codigo=XXX es el endpoint más confiable de la API.
# Devuelve el objeto Comprador completo con:
#   - CodigoOrganismo
#   - NombreOrganismo
#   - RegionUnidad
# Son exactamente 19 requests — completamente manejable.
# ─────────────────────────────────────────────

@retry(
    stop=stop_after_attempt(4),
    wait=wait_exponential(multiplier=2, min=3, max=30),
    retry=retry_if_exception_type(
        (requests.exceptions.Timeout, requests.exceptions.ConnectionError)
    ),
    before_sleep=before_sleep_log(log, logging.WARNING),
    reraise=False,
)
def _fetch_detalle(codigo: str) -> dict | None:
    """Consulta el detalle de una licitación por código."""
    url    = "https://api.mercadopublico.cl/servicios/v1/publico/licitaciones.json"
    params = {"codigo": codigo, "ticket": API_KEY}
    r      = requests.get(url, params=params, timeout=TIMEOUT_SEC)

    if 400 <= r.status_code < 500:
        log.error(f"HTTP {r.status_code} para codigo={codigo} (no reintentable)")
        return None

    r.raise_for_status()
    return r.json()


def extraer_comprador(codigo: str, prefijo: str) -> dict:
    """
    Consulta ?codigo=XXX y extrae CodigoOrganismo, NombreOrganismo
    y RegionUnidad desde el campo Comprador.

    Args:
        codigo:  CodigoExterno representativo del prefijo
        prefijo: prefijo numérico del organismo

    Returns:
        dict auditado con los datos del comprador y status
    """
    base = {
        "prefijo":          prefijo,
        "codigo_consultado": codigo,
        "codigo_organismo": None,
        "nombre_organismo": None,
        "region_unidad":    None,
    }

    try:
        data = _fetch_detalle(codigo)
    except Exception as e:
        return {**base, "status": "error_timeout", "detalle_error": str(e)[:200]}

    if data is None:
        return {**base, "status": "error_http4xx", "detalle_error": "HTTP 4xx"}

    listado = data.get("Listado", [])

    if not listado:
        return {**base, "status": "vacio", "detalle_error": f"Listado vacío para codigo={codigo}"}

    # Extraer Comprador del primer registro
    comprador = listado[0].get("Comprador") or {}

    codigo_org = comprador.get("CodigoOrganismo")
    nombre_org = comprador.get("NombreOrganismo")
    region     = comprador.get("RegionUnidad")

    if not any([codigo_org, nombre_org, region]):
        return {**base, "status": "comprador_vacio", "detalle_error": "Comprador sin campos"}

    return {
        **base,
        "codigo_organismo": str(codigo_org).strip() if codigo_org else None,
        "nombre_organismo": str(nombre_org).strip() if nombre_org else None,
        "region_unidad":    str(region).strip()     if region     else None,
        "status":           "ok",
        "detalle_error":    None,
    }

In [ ]:
# ─────────────────────────────────────────────
# PASO 3: CONSULTAR LOS 19 PREFIJOS
# ─────────────────────────────────────────────

def enriquecer_prefijos(df_dataset: pd.DataFrame) -> pd.DataFrame:
    """
    Itera sobre los 19 prefijos, consulta 1 código por prefijo
    y enriquece con CodigoOrganismo, NombreOrganismo y RegionUnidad.

    Returns:
        DataFrame enriquecido con todos los campos y status auditado.
    """
    log.info("── PASO 2: Consultando API — 1 request por prefijo (19 total) ──")

    resultados = []
    total = len(df_dataset)

    for i, row in enumerate(df_dataset.itertuples(), 1):
        log.info(
            f"  [{i:2d}/{total}] Prefijo {row.prefijo} | "
            f"código: {row.codigo_representativo}"
        )

        resultado = extraer_comprador(row.codigo_representativo, row.prefijo)

        icono = "OK" if resultado["status"] == "ok" else "X"
        log.info(
            f"          [{icono}] {resultado['status']} | "
            f"{(resultado['nombre_organismo'] or '—')[:50]} | "
            f"{(resultado['region_unidad'] or '—')[:40]}"
        )

        resultados.append(resultado)
        time.sleep(SLEEP_SEC)

    df_resultados = pd.DataFrame(resultados)

    # Unir con conteo
    df_merged = df_dataset[["prefijo", "total_le_adjudicadas"]].merge(
        df_resultados, on="prefijo", how="left"
    )

    # Resumen
    resumen = df_merged["status"].value_counts().to_dict()
    log.info(f"\nResumen consultas API:")
    for status, count in sorted(resumen.items(), key=lambda x: -x[1]):
        log.info(f"  {status:<25}: {count}")

    return df_merged

In [ ]:
# ─────────────────────────────────────────────
# PIPELINE PRINCIPAL
# ─────────────────────────────────────────────

def run_pipeline(dry_run: bool = False) -> pd.DataFrame:
    """
    Ejecuta el pipeline completo.

    Args:
        dry_run: Si True, ejecuta todo pero no escribe en GCS.

    Returns:
        DataFrame curated con los 19 organismos enriquecidos.
    """
    log.info("═" * 60)
    log.info("PIPELINE: Enriquecimiento organismos LE — 19 prefijos")
    log.info("═" * 60)

    # ── PASO 1: Dataset filtrado desde GCS ──────────────────────────────────
    df_dataset = construir_dataset_filtrado()

    if df_dataset.empty:
        log.warning("Sin datos. Verificar GCS y filtros.")
        return pd.DataFrame()

    # ── PASO 2: Enriquecimiento desde API (19 requests) ─────────────────────
    df_enriquecido = enriquecer_prefijos(df_dataset)

    # ── Separar capas ────────────────────────────────────────────────────────

    # Curated: registros con nombre y región resueltos
    df_curated = (
        df_enriquecido[df_enriquecido["status"] == "ok"][[
            "prefijo",
            "codigo_organismo",
            "nombre_organismo",
            "region_unidad",
            "total_le_adjudicadas",
        ]]
        .sort_values("total_le_adjudicadas", ascending=False)
        .reset_index(drop=True)
    )

    # Audit: todos los prefijos con trazabilidad completa
    df_audit = (
        df_enriquecido[[
            "prefijo",
            "codigo_consultado",
            "codigo_organismo",
            "nombre_organismo",
            "region_unidad",
            "total_le_adjudicadas",
            "status",
            "detalle_error",
        ]]
        .sort_values("total_le_adjudicadas", ascending=False)
        .reset_index(drop=True)
    )

    # ── Preview ──────────────────────────────────────────────────────────────
    log.info(f"\n{'═'*60}")
    log.info("RESULTADO CURATED:")
    log.info(f"{'═'*60}")
    log.info(f"\n{df_curated.to_string(index=False)}\n")

    # ── Guardar en GCS ───────────────────────────────────────────────────────
    if not dry_run:
        run_date = date.today().strftime("%Y-%m-%d")

        upload_parquet_gcs(
            df_curated,
            f"{CURATED_PATH}/fecha={run_date}/organismos_le.parquet"
        )
        upload_parquet_gcs(
            df_audit,
            f"{AUDIT_PATH}/fecha={run_date}/audit_organismos_le.parquet"
        )
        log.info("Datos guardados en GCS correctamente.")
    else:
        log.info("[DRY RUN] No se escribió en GCS.")

    log.info(f"\nPipeline completado: {len(df_curated)}/{len(df_enriquecido)} organismos en curated.")
    return df_curated

In [ ]:
# ─────────────────────────────────────────────
# ENTRYPOINT
# ─────────────────────────────────────────────

if __name__ == "__main__":
    df_final = run_pipeline(dry_run=False)

04:08:04 [INFO] ════════════════════════════════════════════════════════════
04:08:04 [INFO] PIPELINE: Enriquecimiento organismos LE — 19 prefijos
04:08:04 [INFO] ════════════════════════════════════════════════════════════
04:08:04 [INFO] ── PASO 1: Cargando y filtrando dataset desde GCS ──
04:08:30 [INFO]   2020: 104,527 filas
04:08:52 [INFO]   2021: 127,624 filas
04:09:12 [INFO]   2022: 151,006 filas
04:09:20 [INFO]   2023: 160,789 filas
04:09:23 [INFO]   2024: 168,818 filas
04:09:27 [INFO]   2025: 113,548 filas
04:09:27 [INFO] Total combinado: 826,312 licitaciones
04:09:30 [INFO] Licitaciones LE adjudicadas en prefijos de interés: 4,083
04:09:30 [INFO] 
Conteo por prefijo:
prefijo  total_le_adjudicadas
   3656                   495
   3794                   284
   1660                   279
   3863                   254
   4809                   248
1247197                   220
   3709                   207
    979                   207
   3510                   204
   1509       

In [ ]:
df_final.head()

,prefijo,codigo_organismo,nombre_organismo,region_unidad,total_le_adjudicadas
0,3656,116215,I MUNICIPALIDAD DE REQUINOA,Región del Libertador General Bernardo O´Higgins,495
1,3794,113812,DIRECCION DE LOGISTICA DE CARABINEROS,Región Metropolitana de Santiago,284
2,1660,7323,SERVICIO SALUD COQUIMBO HOSP DE SALAMANC,Región de Coquimbo,279
3,3863,117584,I MUNICIPALIDAD DE SANTA CRUZ,Región del Libertador General Bernardo O´Higgins,254
4,4809,7248,MINISTERIO DE OBRAS PUBLICAS DIREC CION GRAL D...,Región de la Araucanía,248
